Star Schema + Analytics

In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

In [3]:
# Create tables
cursor.executescript('''
CREATE TABLE product (
    product_id INTEGER PRIMARY KEY,
    product_name TEXT,
    category TEXT
);

CREATE TABLE customer (
    customer_id INTEGER PRIMARY KEY,
    customer_name TEXT,
    city TEXT
);

CREATE TABLE date_dim (
    date_id INTEGER PRIMARY KEY,
    full_date TEXT,
    month INTEGER,
    year INTEGER
);

CREATE TABLE sales (
    sales_id INTEGER PRIMARY KEY,
    product_id INTEGER,
    customer_id INTEGER,
    date_id INTEGER,
    amount REAL,
    quantity INTEGER
);
''')
conn.commit()

In [4]:
# Insert sample data
cursor.executemany("INSERT INTO product VALUES (?, ?, ?)", [
    (1, 'Apple', 'Fruit'),
    (2, 'Banana', 'Fruit'),
    (3, 'Carrot', 'Vegetable')
])

cursor.executemany("INSERT INTO customer VALUES (?, ?, ?)", [
    (1, 'Alice', 'Delhi'),
    (2, 'Bob', 'Mumbai'),
    (3, 'Charlie', 'Delhi')
])

cursor.executemany("INSERT INTO date_dim VALUES (?, ?, ?, ?)", [
    (1, '2024-01-01', 1, 2024),
    (2, '2024-02-01', 2, 2024),
    (3, '2024-03-01', 3, 2024)
])

cursor.executemany("INSERT INTO sales VALUES (?, ?, ?, ?, ?, ?)", [
    (1, 1, 1, 1, 100.0, 2),
    (2, 2, 2, 2, 150.0, 3),
    (3, 3, 3, 3, 200.0, 5),
    (4, 1, 2, 1, 120.0, 1),
    (5, 2, 1, 2, 180.0, 4)
])

conn.commit()

In [5]:
# Join dataset
query = '''
SELECT p.product_name, c.customer_name, c.city, d.month, d.year, s.amount, s.quantity
FROM sales s
JOIN product p ON s.product_id = p.product_id
JOIN customer c ON s.customer_id = c.customer_id
JOIN date_dim d ON s.date_id = d.date_id
'''

df = pd.read_sql_query(query, conn)
df

,product_name,customer_name,city,month,year,amount,quantity
0,Apple,Alice,Delhi,1,2024,100.0,2
1,Banana,Bob,Mumbai,2,2024,150.0,3
2,Carrot,Charlie,Delhi,3,2024,200.0,5
3,Apple,Bob,Mumbai,1,2024,120.0,1
4,Banana,Alice,Delhi,2,2024,180.0,4


## Analytics

In [6]:
# Total sales per product
pd.read_sql_query('''
SELECT p.product_name, SUM(s.amount) AS total_sales
FROM sales s
JOIN product p ON s.product_id = p.product_id
GROUP BY p.product_name
''', conn)

,product_name,total_sales
0,Apple,220.0
1,Banana,330.0
2,Carrot,200.0


In [7]:
# Total sales per customer
pd.read_sql_query('''
SELECT c.customer_name, SUM(s.amount) AS total_sales
FROM sales s
JOIN customer c ON s.customer_id = c.customer_id
GROUP BY c.customer_name
''', conn)

,customer_name,total_sales
0,Alice,280.0
1,Bob,270.0
2,Charlie,200.0


In [8]:
# Total sales per city
pd.read_sql_query('''
SELECT c.city, SUM(s.amount) AS total_sales
FROM sales s
JOIN customer c ON s.customer_id = c.customer_id
GROUP BY c.city
''', conn)

,city,total_sales
0,Delhi,480.0
1,Mumbai,270.0


In [9]:
# Total sales per month
pd.read_sql_query('''
SELECT d.year, d.month, SUM(s.amount) AS total_sales
FROM sales s
JOIN date_dim d ON s.date_id = d.date_id
GROUP BY d.year, d.month
ORDER BY d.year, d.month
''', conn)

,year,month,total_sales
0,2024,1,220.0
1,2024,2,330.0
2,2024,3,200.0


In [10]:
# Top product
pd.read_sql_query('''
SELECT p.product_name, SUM(s.amount) AS total_sales
FROM sales s
JOIN product p ON s.product_id = p.product_id
GROUP BY p.product_name
ORDER BY total_sales DESC
LIMIT 1
''', conn)

,product_name,total_sales
0,Banana,330.0


In [11]:
# Top customer
pd.read_sql_query('''
SELECT c.customer_name, SUM(s.amount) AS total_sales
FROM sales s
JOIN customer c ON s.customer_id = c.customer_id
GROUP BY c.customer_name
ORDER BY total_sales DESC
LIMIT 1
''', conn)

,customer_name,total_sales
0,Alice,280.0


In [12]:
# Average sales per transaction
pd.read_sql_query('SELECT AVG(amount) AS avg_sales FROM sales', conn)

,avg_sales
0,150.0


In [13]:
# Total quantity per product
pd.read_sql_query('''
SELECT p.product_name, SUM(s.quantity) AS total_quantity
FROM sales s
JOIN product p ON s.product_id = p.product_id
GROUP BY p.product_name
''', conn)

,product_name,total_quantity
0,Apple,3
1,Banana,7
2,Carrot,5
